# Sentiment Analysis with Deep Learning using BERT

## Libraries

In [1]:
import torch
import random
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import f1_score
from transformers import BertTokenizer

from torch.optim import AdamW
from torch.utils.data import TensorDataset
from sklearn.model_selection import train_test_split
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler

BASE_DIR = Path.cwd()
if not (BASE_DIR / "data").exists() and (BASE_DIR / "notebooks" / "data").exists():
    BASE_DIR = BASE_DIR / "notebooks"

DATA_DIR = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

/Users/sergio.fernandez/Documents/personal/projects/robotics/transformer-studio/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Task 1: Introduction

### What is BERT

BERT is a large-scale transformer-based Language Model that can be finetuned for a variety of tasks.

For more information, the original paper can be found [here](https://arxiv.org/abs/1810.04805). 

[HuggingFace documentation](https://huggingface.co/transformers/model_doc/bert.html)

[Bert documentation](https://characters.fandom.com/wiki/Bert_(Sesame_Street) ;)

<img src="images/BERT_diagrams.png" width="1000">

## Task: Exploratory Data Analysis and Preprocessing

In [2]:
df = pd.read_csv(
    DATA_DIR / "sentiment_analysis" / "smile-annotations-final.csv",
    names=["id", "text", "category"],
)
df.set_index("id", inplace=True)

In [3]:
df.head()

,text,category
id,,
611857364396965889,@aandraous @britishmuseum @AndrewsAntonio Merc...,nocode
614484565059596288,Dorian Gray with Rainbow Scarf #LoveWins (from...,happy
614746522043973632,@SelectShowcase @Tate_StIves ... Replace with ...,happy
614877582664835073,@Sofabsports thank you for following me back. ...,happy
611932373039644672,@britishmuseum @TudorHistory What a beautiful ...,happy


In [4]:
df.category.value_counts()

category
nocode               1572
happy                1137
not-relevant          214
angry                  57
surprise               35
sad                    32
happy|surprise         11
happy|sad               9
disgust|angry           7
disgust                 6
sad|disgust             2
sad|angry               2
sad|disgust|angry       1
Name: count, dtype: int64

In [5]:
print(f"Data size before filtering: {len(df)}")
df = df[~df.category.apply(lambda x: len(x.split("|")) > 1 or x == "nocode")]
print(f"Data size after filtering: {len(df)}")

Data size before filtering: 3085
Data size after filtering: 1481


In [6]:
df.category.value_counts()

category
happy           1137
not-relevant     214
angry             57
surprise          35
sad               32
disgust            6
Name: count, dtype: int64

In [7]:
label_dict = {label: i for i, label in enumerate(df.category.unique())}
label_dict

{'happy': 0,
 'not-relevant': 1,
 'angry': 2,
 'disgust': 3,
 'sad': 4,
 'surprise': 5}

In [8]:
df["label"] = df.category.replace(label_dict)
df.sample(5)

,text,category,label
id,,,
614007730311987200,Ketaki Sheth's photos of the Sidi people are a...,happy,0
614003685287456768,Faience vessel from @TheEES excavations at Ses...,happy,0
613986003922092032,Off to @britishmuseum for #definingbeauty want...,happy,0
614397150391353344,"It's sunny, I was just given an @hummingbbaker...",happy,0
611169604346511361,Lovely reception at the @NationalGallery on Mo...,happy,0


## Task 3: Training/Validation Split

In [9]:
X_train, X_val, y_train, y_val = train_test_split(
    df.index.values,
    df.label.values,
    test_size=0.15,
    random_state=17,
    stratify=df.label.values,
)

In [10]:
df["data_type"] = ["not_set"] * df.shape[0]

In [11]:
df.loc[X_train, "data_type"] = "train"
df.loc[X_val, "data_type"] = "val"

In [12]:
df.groupby(["category", "label", "data_type"]).count()

text
category     label data_type      
angry        2     train        48
                   val           9
disgust      3     train         5
                   val           1
happy        0     train       966
                   val         171
not-relevant 1     train       182
                   val          32
sad          4     train        27
                   val           5
surprise     5     train        30
                   val           5

## Task 4: Loading Tokenizer and Encoding out Data

In [13]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)

In [14]:
encoded_data_train = tokenizer(
    df[df.data_type == "train"].text.astype(str).tolist(),
    add_special_tokens=True,
    return_attention_mask=True,
    padding="max_length",
    truncation=True,
    max_length=256,
    return_tensors="pt",  # pytorch tensors
)

encoded_data_val = tokenizer(
    df[df.data_type == "val"].text.astype(str).tolist(),
    add_special_tokens=True,
    return_attention_mask=True,
    padding="max_length",
    truncation=True,
    max_length=256,
    return_tensors="pt",  # pytorch tensors
)

input_ids_train = encoded_data_train["input_ids"]
attention_masks_train = encoded_data_train["attention_mask"]
labels_train = torch.tensor(
    df[df.data_type == "train"].label.apply(lambda x: int(x)).values
)

input_ids_val = encoded_data_val["input_ids"]
attention_masks_val = encoded_data_val["attention_mask"]
labels_val = torch.tensor(
    df[df.data_type == "val"].label.apply(lambda x: int(x)).values
)

In [15]:
dataset_train = TensorDataset(input_ids_train, attention_masks_train, labels_train)
dataset_val = TensorDataset(input_ids_val, attention_masks_val, labels_val)

print(f"Training data size: {len(dataset_train)}")
print(f"Validation data size: {len(dataset_val)}")

Training data size: 1258
Validation data size: 223


## Task 5 setting up BERT Pretrained Model

There are several related ways to load BERT depending on the task:

  | Class | What it loads | When to use it |
  | --- | --- | --- |
  | BertModel | The base BERT encoder only, without a task-specific head. | Use it when you want embeddings or when you plan to build your own custom head manually. |
  | **BertForSequenceClassification** | BERT plus a sequence-level classification head. | Best choice for sentiment analysis, topic classification, and any task with one label per full sentence or document. |
  | AutoModel | The base encoder chosen automatically from the checkpoint configuration. | Same idea as BertModel, but more flexible when you want code that can swap to another architecture later. |
  | AutoModelForSequenceClassification | A sequence-classification model chosen automatically from the checkpoint configuration. | Same task as BertForSequenceClassification, but better if you may later replace BERT with RoBERTa, DistilBERT, DeBERTa, etc. |
  | BertForTokenClassification | BERT plus a token-level classifier. | Use it for NER, POS tagging, or any task where every token needs its own label. |
  | BertForQuestionAnswering | BERT plus a head that predicts the start and end positions of an answer span. | Use it for extractive question answering tasks. |
  | BertForMaskedLM | BERT plus the masked-language-model head. | Use it for fill-mask tasks or for continuing masked-language-model pretraining. |
  | BertForPreTraining | BERT plus the original pretraining heads. | Use it when you want the original BERT pretraining setup, which combines masked language modeling and next sentence prediction. |
  | BertForMultipleChoice | BERT plus a multiple-choice classification head. | Use it when each example contains several candidate answers and the model must choose one. |
  | BertForNextSentencePrediction | BERT plus a binary next-sentence-prediction head. | Use it only for the specific NSP objective; it is much narrower than full sequence classification. |
  | BertLMHeadModel | BERT plus a language-model head. | This is more specialized and less common in standard classification notebooks. |

  Two extra details matter when loading any of these classes:

  - .from_pretrained(...) loads pretrained weights and is the standard option for fine-tuning.
  - Building the model from a config creates the architecture but not the pretrained weights, so it starts from random initialization.

  For this notebook, the most direct options are BertForSequenceClassification if we want to stay explicit, or AutoModelForSequenceClassification if we want the
  code to remain model-agnostic.

In [16]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_dict),
    output_attentions=False,
    output_hidden_states=False,
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9474.83it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

## Task 6: Creating Data Loaders

In [17]:
batch_size = 32

dataloader_train = DataLoader(
    dataset_train, sampler=RandomSampler(dataset_train), batch_size=batch_size
)

dataloader_val = DataLoader(
    dataset_val, sampler=SequentialSampler(dataset_val), batch_size=batch_size
)

## Task 7: Setting Up Optimizer and Scheduler

In [18]:
optimizer = AdamW(model.parameters(), lr=1e-5, eps=1e-8)

In [19]:
EPOCHS = 10
NUM_TRAIN_STEPS = len(dataloader_train) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=0, num_training_steps=NUM_TRAIN_STEPS
)

## Task 8: Defining our Performance Metrics

In [20]:
def f1_score_func(preds, labels):
    preds_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return f1_score(labels_flat, preds_flat, average="weighted")

In [21]:
def accuracy_per_class(preds, labels):
    label_dict_inverse = {v: k for k, v in label_dict.items()}
    preds_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    for label in np.unique(labels_flat):
        y_preds = preds_flat[labels_flat == label]
        y_true = labels_flat[labels_flat == label]
        print(
            f"Class: {label_dict_inverse[label]}",
            f"Accuracy: {len(y_preds[y_preds == label])}/{len(y_true)} = {len(y_preds[y_preds == label]) / len(y_true)}",
        )

## Task 9: Creating our Training Loop

In [22]:
seed_val = 17
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)

In [23]:
device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    print("No GPU available, using the CPU instead.")


device = torch.device(device)
device

device(type='mps')

In [24]:
model = model.to(device)
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [25]:
def evaluate(dataloader_val):

    model.eval()
    
    loss_val_total = 0
    predictions, true_vals = [], []
    
    for batch in dataloader_val:
        
        batch = tuple(b.to(device) for b in batch)
        
        inputs = {'input_ids':      batch[0],
                  'attention_mask': batch[1],
                  'labels':         batch[2],
                 }

        with torch.no_grad():        
            outputs = model(**inputs)
            
        loss = outputs[0]
        logits = outputs[1]
        loss_val_total += loss.item()

        logits = logits.detach().cpu().numpy()
        label_ids = inputs['labels'].cpu().numpy()
        predictions.append(logits)
        true_vals.append(label_ids)
    
    loss_val_avg = loss_val_total/len(dataloader_val) 
    
    predictions = np.concatenate(predictions, axis=0)
    true_vals = np.concatenate(true_vals, axis=0)
            
    return loss_val_avg, predictions, true_vals

In [26]:
for epoch in tqdm(range(1, EPOCHS+1)):
    model.train()
    
    loss_train_total = 0
    
    progress_bar = tqdm(dataloader_train, desc=f"Epoch {epoch}", leave=False, disable=False)
    for batch in progress_bar:
        
        model.zero_grad()
        
        batch = tuple(b.to(device) for b in batch)
        
        inputs = {'input_ids':      batch[0],
                  'attention_mask': batch[1],
                  'labels':         batch[2],
                 }        
        
        outputs = model(**inputs)
        
        loss = outputs[0]
        loss_train_total += loss.item()
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        optimizer.step()
        scheduler.step()
        
        progress_bar.set_postfix(
            {'training_loss': '{:.3f}'.format(loss.item()/len(batch))}
            )
    
    torch.save(model.state_dict(), MODELS_DIR / f"finetuned_BERT_epoch_{epoch}.model")

    tqdm.write(f'\nEpoch {epoch}')
    loss_train_avg = loss_train_total/len(dataloader_train)
    tqdm.write(f'Training loss: {loss_train_avg}')
    loss_val_avg, predictions, true_vals = evaluate(dataloader_val)
    val_f1 = f1_score_func(predictions, true_vals)
    tqdm.write(f'Validation loss: {loss_val_avg}')
    tqdm.write(f'F1 Score (Weighted): {val_f1}')




  0%|          | 0/10 [00:51<?, ?it/s]


Epoch 1
Training loss: 1.0721241697669028


 10%|█         | 1/10 [00:53<08:04, 53.82s/it]

Validation loss: 0.8040845223835537
F1 Score (Weighted): 0.6656119824269878


 10%|█         | 1/10 [01:44<08:04, 53.82s/it]

...

Validation loss: 0.549803940313203
F1 Score (Weighted): 0.8312738277219534


 90%|█████████ | 9/10 [09:12<00:55, 55.70s/it]


Epoch 10
Training loss: 0.19701033215969802


100%|██████████| 10/10 [09:15<00:00, 55.53s/it]

Validation loss: 0.5538882102285113
F1 Score (Weighted): 0.8312324119104262


In [27]:
_, predictions, true_vals = evaluate(dataloader_val)
accuracy_per_class(predictions, true_vals)

Class: happy Accuracy: 162/171 = 0.9473684210526315
Class: not-relevant Accuracy: 18/32 = 0.5625
Class: angry Accuracy: 7/9 = 0.7777777777777778
Class: disgust Accuracy: 0/1 = 0.0
Class: sad Accuracy: 0/5 = 0.0
Class: surprise Accuracy: 2/5 = 0.4
